# YOLOv8l 인스턴스 분할 학습

- 설명: YOLOv8l-seg 기반 안전고리 인스턴스 분할 실험입니다.
- 작성자: 김동혁

> 공개 저장소에는 데이터, 모델 가중치, API 키와 노트북 실행 결과를 포함하지 않습니다.


# YOLOv8 Instance Segmentation 학습 코드

In [ ]:
# =========================
# 1. 환경 확인
# =========================

import torch

print("CUDA available:", torch.cuda.is_available())

if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))
else:
    print("GPU를 사용할 수 없습니다. Colab 런타임 유형을 GPU로 변경하세요.")

In [ ]:
# =========================
# 2. 패키지 설치
# =========================

!pip install -q roboflow ultralytics

In [ ]:
# =========================
# 3. Roboflow 데이터셋 다운로드
# =========================

import os
from roboflow import Roboflow

ROBOFLOW_API_KEY = os.environ.get("ROBOFLOW_API_KEY")
if not ROBOFLOW_API_KEY:
    raise RuntimeError("ROBOFLOW_API_KEY 환경변수가 설정되지 않았습니다.")
rf = Roboflow(api_key=ROBOFLOW_API_KEY)
project = rf.workspace("s-workspace-o2evy").project("dyd-safe-firrr")
version = project.version(1)
dataset = version.download("yolov8")

print("Dataset location:", dataset.location)

In [ ]:
# =========================
# 4. data.yaml 확인
# =========================

import os

data_yaml_path = os.path.join(dataset.location, "data.yaml")

print("data.yaml:", data_yaml_path)

with open(data_yaml_path, "r", encoding="utf-8") as f:
    print(f.read())

In [ ]:
# =========================
# 5. YOLOv8 Segmentation 모델 학습
# =========================

from ultralytics import YOLO

model = YOLO("yolov8l-seg.pt")

results = model.train(
    data=data_yaml_path,
    epochs=100,
    imgsz=1024,
    batch=8,
    patience=20,
    device=0,
    workers=2,
    optimizer="auto",
    cos_lr=True,
    name="hook_yolov8l_seg"
)

# 학습 결과 확인

In [ ]:
# =========================
# 6. 학습 결과 이미지 확인
# =========================

from IPython.display import Image, display
import os

result_dir = "runs/segment/hook_yolov8l_seg"

result_images = [
    "results.png",
    "confusion_matrix.png",
    "MaskP_curve.png",
    "MaskR_curve.png",
    "MaskF1_curve.png",
    "MaskPR_curve.png",
]

for img_name in result_images:
    img_path = os.path.join(result_dir, img_name)
    if os.path.exists(img_path):
        print(img_name)
        display(Image(filename=img_path))

In [ ]:
# =========================
# 7. 검증 데이터 평가
# =========================

from ultralytics import YOLO
import os

best_model_path = "runs/segment/hook_yolov8l_seg/weights/best.pt"

trained_model = YOLO(best_model_path)

metrics = trained_model.val(
    data=data_yaml_path,
    imgsz=1024,
    device=0
)

print(metrics)

# 새 이미지로 인스턴스 세그멘테이션 테스트

In [ ]:
# =========================
# 8. 새 이미지 업로드
# =========================

from google.colab import files

uploaded = files.upload()
uploaded_image_paths = list(uploaded.keys())

print("Uploaded images:", uploaded_image_paths)

In [ ]:
# =========================
# 9. 새 이미지 예측
# =========================

results = trained_model.predict(
    source=uploaded_image_paths,
    imgsz=1024,
    conf=0.25,
    save=True
)

print("Prediction completed.")

In [ ]:
# =========================
# 10. 예측 결과 확인
# =========================

import glob
import os
from IPython.display import Image, display

predict_dirs = sorted(glob.glob("runs/segment/predict*"))
latest_predict_dir = predict_dirs[-1]

result_images = glob.glob(os.path.join(latest_predict_dir, "*.jpg")) + \
                glob.glob(os.path.join(latest_predict_dir, "*.jpeg")) + \
                glob.glob(os.path.join(latest_predict_dir, "*.png"))

print("Result dir:", latest_predict_dir)

for img_path in result_images:
    display(Image(filename=img_path))

# 학습된 모델 다운로드

In [ ]:
# =========================
# 11. best.pt 다운로드
# =========================

from google.colab import files

best_model_path = "runs/segment/hook_yolov8l_seg/weights/best.pt"

if os.path.exists(best_model_path):
    files.download(best_model_path)
else:
    print("best.pt 파일을 찾을 수 없습니다:", best_model_path)

# files.download(best_model_path)

In [ ]:
# =========================
# 12. last.pt 다운로드
# =========================

from google.colab import files

last_model_path = "runs/segment/hook_yolov8l_seg/weights/last.pt"

if os.path.exists(last_model_path):
    files.download(last_model_path)
else:
    print("last.pt 파일을 찾을 수 없습니다:", last_model_path)

# last_model_path = "runs/segment/hook_yolov8l_seg/weights/last.pt"

In [ ]:
# 1. 구글 드라이브 마운트 (최초 1회 인증 팝업 확인 필요)
from google.colab import drive

drive.mount('/content/drive')

In [ ]:
# ==========================================
# 13. 학습 결과 폴더 전체 구글 드라이브로 백업
# ==========================================
import shutil
import os


# 원본 결과 폴더 및 구글 드라이브 저장 경로 설정
source_dir = "runs/segment/hook_yolov8l_seg"
drive_dir = "/content/drive/MyDrive/YOLOv8L_Training_Results"

# 2. 폴더 통째로 복사
if os.path.exists(source_dir):
    shutil.copytree(source_dir, drive_dir, dirs_exist_ok=True)
    print(f"📦 백업 완료! 구글 드라이브의 '{drive_dir}' 폴더에서 best.pt, last.pt 및 모든 평가지표를 확인하세요.")
else:
    print("❌ 원본 결과 폴더를 찾을 수 없습니다. 학습이 정상적으로 완료되었는지 확인해 주세요.")

In [ ]:
# ==========================================
# 전체 결과 폴더 로컬 PC로 다운로드 (압축 후 다운)
# ==========================================
import shutil
from google.colab import files
import os

# 1. 압축할 원본 폴더 경로 (학습한 모델 폴더명으로 확인/수정)
source_dir = "runs/segment/hook_yolov8l_seg"

# 2. 생성될 압축 파일의 이름 (확장자 .zip은 제외하고 입력)
zip_filename = "YOLOv8L_Training_Results"

if os.path.exists(source_dir):
    # 폴더를 .zip 파일로 압축
    print("📦 폴더 압축을 시작합니다... (데이터 크기에 따라 다소 시간이 걸립니다)")
    shutil.make_archive(zip_filename, 'zip', source_dir)

    # 압축된 파일을 로컬 PC로 다운로드
    print("⬇️ 로컬 다운로드를 요청합니다. 브라우저의 다운로드 팝업을 확인해 주세요!")
    files.download(f"{zip_filename}.zip")
else:
    print("❌ 원본 결과 폴더를 찾을 수 없습니다. 학습 경로를 다시 한번 확인해 주세요.")

### 저장된 모델 결과 확인

In [ ]:
# =========================
# 1. 패키지 설치
# =========================

!pip install -q ultralytics

In [ ]:
# =========================
# 2. 저장된 모델 업로드
# =========================

from google.colab import files

uploaded = files.upload()
model_path = list(uploaded.keys())[0]

print("Model path:", model_path)

In [ ]:
# =========================
# 3. 모델 불러오기
# =========================

from ultralytics import YOLO

model = YOLO(model_path)

In [ ]:
# =========================
# 4. 테스트 이미지 업로드
# =========================

uploaded_images = files.upload()
image_paths = list(uploaded_images.keys())

print("Test images:", image_paths)

In [ ]:
# =========================
# 5. 예측 실행
# =========================

results = model.predict(
    source=image_paths,
    imgsz=1024,
    conf=0.1,
    save=True
)

print("Prediction completed.")

In [ ]:
# =========================
# 6. 결과 이미지 확인
# =========================

import glob
import os
from IPython.display import Image, display

# Detection 모델이면 runs/detect/predict*
# Segmentation 모델이면 runs/segment/predict*
predict_dirs = sorted(glob.glob("runs/detect/predict*")) + \
               sorted(glob.glob("runs/segment/predict*"))

latest_predict_dir = predict_dirs[-1]

result_images = glob.glob(os.path.join(latest_predict_dir, "*.jpg")) + \
                glob.glob(os.path.join(latest_predict_dir, "*.jpeg")) + \
                glob.glob(os.path.join(latest_predict_dir, "*.png"))

print("Result dir:", latest_predict_dir)

for img_path in result_images:
    display(Image(filename=img_path))